# 🏥 Vietnamese Medical NER — LoRA Fine-Tuning (Kaggle)

Train Qwen2.5-7B-Instruct (or Vistral-7B) with QLoRA on `bootstrap_gt/` (100 samples).

**Setup on Kaggle:**
1. Add dataset `vtai-race` containing `bootstrap_gt/` and `input/input/` folders
2. Enable GPU: Settings → Accelerator → GPU T4 x2 (or P100)
3. Run all cells. LoRA adapter saved to `/kaggle/working/lora_adapter/`.

**Approx time:** 20-40 min on T4 x2 for 5 epochs.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
!pip install -q editdistance

## 2. Verify environment

In [ ]:
import torch, os, sys
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Python: {sys.version}")

## 3. Copy project files (or git clone)

In [ ]:
# Adjust based on your setup
# Option A: dataset uploaded to Kaggle
WORK_DIR = '/kaggle/working'
DATA_ROOT = '/kaggle/input/vtai-race'  # uploaded dataset

# Option B: git clone (if private repo)
# !git clone https://github.com/YOUR_USER/vtai-race.git /kaggle/working/vtai-race
# WORK_DIR = '/kaggle/working/vtai-race'
# DATA_ROOT = WORK_DIR

os.chdir(WORK_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"\nContents: {os.listdir(WORK_DIR)[:10]}")
print(f"\nData contents: {os.listdir(DATA_ROOT)[:10] if os.path.exists(DATA_ROOT) else 'NOT FOUND'}")

## 4. Choose base model

| Model | VRAM needed | Vietnamese | Notes |
|-------|-------------|------------|-------|
| `Qwen/Qwen2.5-7B-Instruct` | ~16GB | Good | Best balance |
| `Viet-Mistral/Vistral-7B-chat` | ~16GB | Best | Slightly weaker JSON |
| `Qwen/Qwen2.5-3B-Instruct` | ~8GB (4-bit) | OK | Fast, smaller |

Default: Qwen2.5-7B-Instruct with QLoRA (4-bit).

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # change to test others
# BASE_MODEL = "Viet-Mistral/Vistral-7B-chat"
# BASE_MODEL = "Intelligent-Internet/II-Medical-8B"  # NOT recommended for VN NER
USE_QLORA = True
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 5
LR = 1e-4
MAX_SEQ_LEN = 4096
BATCH_SIZE = 1
GRAD_ACCUM = 8
OUTPUT_DIR = '/kaggle/working/lora_adapter'
print(f"Training {BASE_MODEL} with {'QLoRA' if USE_QLORA else 'LoRA'} for {EPOCHS} epochs")

## 5. Copy project code to working dir

In [ ]:
# Copy needed Python files to /kaggle/working so the training script can find them
import shutil
src_files = ['llm_prompts.py', 'sft_train_lora.py', 'llm_ner_config.py', 'drug_dict_v3.py', 'vocab_v5.py']
for f in src_files:
    src = f'{DATA_ROOT}/{f}'
    dst = f'/kaggle/working/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {f}")
    else:
        print(f"⚠ Not found: {src}")

# Symlink bootstrap_gt and input
for d in ['bootstrap_gt', 'input']:
    src = f'{DATA_ROOT}/{d}'
    dst = f'/kaggle/working/{d}'
    if os.path.exists(src):
        if os.path.exists(dst):
            if os.path.islink(dst): os.unlink(dst)
            else: shutil.rmtree(dst)
        os.symlink(src, dst)
        print(f"Linked {d}")
    else:
        print(f"⚠ Not found: {src}")

## 6. Run training

In [ ]:
cmd = f"""python sft_train_lora.py \\
  --base_model {BASE_MODEL} \\
  --gt_dir bootstrap_gt \\
  --input_dir input/input \\
  --output_dir {OUTPUT_DIR} \\
  --epochs {EPOCHS} \\
  --lr {LR} \\
  --lora_r {LORA_R} \\
  --lora_alpha {LORA_ALPHA} \\
  --max_seq_len {MAX_SEQ_LEN} \\
  --batch_size {BATCH_SIZE} \\
  --grad_accum {GRAD_ACCUM} \\
  {"--qlora" if USE_QLORA else ""}\\
  --warmup_ratio 0.05
"""
print("Running:\n", cmd)
%time $cmd

## 7. Verify saved adapter

In [ ]:
import os
print(f"\n=== Adapter files ===")
for f in os.listdir(OUTPUT_DIR):
    sz = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f"  {f}: {sz:.1f} KB")

# Check adapter size — typically 20-50MB for 7B with r=16

## 8. Test inference (sanity check)

In [ ]:
import sys, json
sys.path.insert(0, '/kaggle/working')

from llm_inference import LLMExtractor
from peft import PeftModel
import torch

# Load base model in 4-bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, OUTPUT_DIR)
model.eval()

# Quick test on first input
test_text = open('input/input/1.txt', encoding='utf-8').read()[:2000]  # first half
from llm_prompts import build_chat_messages, SYSTEM_PROMPT
msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Trích xuất thực thể y tế từ:\n{test_text}\n\nJSON:"},
]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=False, repetition_penalty=1.05)
gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n=== LLM output ===\n", gen[:2000])